# Ames Housing Price Prediction – LASSO Regression

This notebook demonstrates a structured machine learning workflow using regularized regression for housing price prediction.

**Dataset:** Ames Housing Dataset (Kaggle)  
**Goal:** Predict `SalePrice` using LASSO regression with cross-validated hyperparameter tuning.

## Modeling Strategy

This notebook follows a structured ML workflow:
- Data cleaning & missing value imputation
- Ordinal feature encoding
- Outlier detection and removal
- Log-transformed regression target
- LASSO regularization with cross-validated grid search
- Evaluation on original price scale (RMSE, R²)
- Feature importance visualization

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

## 1. Load Data

In [ ]:
# Load data
# Download train.csv and test.csv from:
# https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Flag to track origin after combining
train["train_flag"] = 1
test["train_flag"] = 0

# Combine for consistent preprocessing
df = pd.concat([train, test], axis=0, ignore_index=True)

print("Combined shape:", df.shape)
print("Train rows:", train.shape[0])
print("Test rows:", test.shape[0])

## 2. Missing Value Handling

In [ ]:
def handle_missing_values(df):
    """
    Imputes missing values using domain knowledge:
    - Categorical absence fields (e.g. no pool, no fence) → 'None'
    - LotFrontage → neighborhood median
    - Garage/Basement numeric → 0, categorical → 'None'
    - Remaining → training set mode
    """
    train_mask = df["train_flag"] == 1

    # Absence-coded categoricals
    fill_none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu"]
    for col in fill_none_cols:
        if col in df.columns:
            df[col] = df[col].fillna("None")

    # LotFrontage: impute by neighborhood median (train only), then global median
    if "LotFrontage" in df.columns:
        lf_med = df.loc[train_mask].groupby("Neighborhood")["LotFrontage"].median()
        df["LotFrontage"] = df["LotFrontage"].fillna(df["Neighborhood"].map(lf_med))
        df["LotFrontage"] = df["LotFrontage"].fillna(df.loc[train_mask, "LotFrontage"].median())

    # Garage columns
    garage_cols = [c for c in df.columns if c.startswith("Garage")]
    for col in garage_cols:
        if df[col].dtype == "object":
            df[col] = df[col].fillna("None")
        else:
            df[col] = df[col].fillna(0)

    # Basement columns
    bsmt_cols = [c for c in df.columns if c.startswith("Bsmt")]
    for col in bsmt_cols:
        if df[col].dtype == "object":
            df[col] = df[col].fillna("None")
        else:
            df[col] = df[col].fillna(0)

    # Fill any remaining missing with training-set mode
    mode_row = df.loc[train_mask].mode().iloc[0]
    df = df.fillna(mode_row)

    print("Remaining missing after imputation:", df.isna().sum().sum())
    return df

## 3. Ordinal Encoding

Quality/condition features are encoded as ordered integers based on domain knowledge rather than one-hot encoding, preserving their natural hierarchy.

In [ ]:
def apply_ordinal_mappings(df):
    """
    Maps ordered categorical features to integer scales.
    E.g. ExterQual: Po=0, Fa=1, TA=2, Gd=3, Ex=4
    """
    quality_map_5 = {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4}
    quality_map_6 = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
    exposure_map  = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
    fin_map       = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
    garage_fin_map = {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}

    mapping_dict = {
        "ExterQual":    quality_map_5,
        "ExterCond":    quality_map_5,
        "HeatingQC":    quality_map_5,
        "KitchenQual":  quality_map_5,
        "BsmtQual":     quality_map_6,
        "BsmtCond":     quality_map_6,
        "GarageQual":   quality_map_6,
        "GarageCond":   quality_map_6,
        "FireplaceQu":  quality_map_6,
        "BsmtExposure": exposure_map,
        "BsmtFinType1": fin_map,
        "BsmtFinType2": fin_map,
        "GarageFinish": garage_fin_map,
    }

    for col, mapping in mapping_dict.items():
        if col in df.columns:
            df[col] = df[col].map(mapping)

    return df

## 4. Outlier Removal

Extreme values in key area/size features are removed from the training set only, based on visual inspection and domain knowledge.

In [ ]:
def remove_outliers(df):
    """
    Removes training-set rows with extreme values in key size features.
    Test set rows are never removed.
    """
    train_mask = df["train_flag"] == 1

    conditions = (
        (df["LotFrontage"] > 200) |
        (df["1stFlrSF"]    > 4000) |
        (df["LotArea"]     > 100000) |
        (df["BsmtFinSF1"]  > 4000) |
        (df["TotalBsmtSF"] > 5000) |
        (df["GrLivArea"]   > 4000)
    )

    drop_index = df[train_mask & conditions].index
    print("Outliers removed:", len(drop_index))

    return df.drop(index=drop_index)

## 5. Apply Preprocessing

In [ ]:
df = handle_missing_values(df)
df = apply_ordinal_mappings(df)
df = remove_outliers(df)

## 6. Target Transformation

`SalePrice` is right-skewed. Applying `log1p` makes the distribution more normal, which benefits linear models.

In [ ]:
# Log-transform target (train set only)
y = np.log1p(df.loc[df["train_flag"] == 1, "SalePrice"])

# Visualize distribution before and after transformation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.loc[df["train_flag"] == 1, "SalePrice"].hist(bins=50, ax=axes[0])
axes[0].set_title("SalePrice (original)")
axes[0].set_xlabel("Price")

y.hist(bins=50, ax=axes[1], color="steelblue")
axes[1].set_title("log(SalePrice + 1)")
axes[1].set_xlabel("log(Price)")

plt.tight_layout()
plt.show()

## 7. Feature Preparation

Remaining categorical features are one-hot encoded. `drop_first=True` avoids perfect multicollinearity (dummy variable trap).

In [ ]:
df_model = df.drop(["SalePrice", "Id"], axis=1, errors="ignore")

categorical_cols = df_model.select_dtypes(include=["object"]).columns

# drop_first=True avoids the dummy variable trap
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

# Split back into train and Kaggle test sets
X        = df_model[df_model["train_flag"] == 1].drop("train_flag", axis=1)
X_kaggle = df_model[df_model["train_flag"] == 0].drop("train_flag", axis=1)

print("Training features shape:", X.shape)
print("Kaggle test features shape:", X_kaggle.shape)

## 8. Train / Validation Split

In [ ]:
# X_val / y_val = held-out validation set (not the Kaggle test set)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])

## 9. Pipeline + GridSearchCV

A `Pipeline` ensures that `StandardScaler` is fit only on training folds during cross-validation, preventing data leakage.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso",  Lasso(max_iter=10000))
])

param_grid = {
    "lasso__alpha": np.logspace(-4, 0, 50)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best alpha:", round(grid.best_params_["lasso__alpha"], 6))
print("Best CV RMSE (log scale):", round(np.sqrt(-grid.best_score_), 5))

## 10. Evaluation on Validation Set

Predictions are converted back from log scale using `expm1` to compute RMSE and R² in original dollar units.

In [ ]:
best_model = grid.best_estimator_

y_pred_log = best_model.predict(X_val)

# Inverse log transform
y_pred_val    = np.expm1(y_pred_log)
y_val_original = np.expm1(y_val)

rmse = np.sqrt(mean_squared_error(y_val_original, y_pred_val))
r2   = r2_score(y_val_original, y_pred_val)

print(f"Validation RMSE: ${rmse:,.0f}")
print(f"Validation R²:   {r2:.4f}")

# Predicted vs Actual plot
plt.figure(figsize=(7, 5))
plt.scatter(y_val_original, y_pred_val, alpha=0.4, edgecolors="k", linewidths=0.3)
plt.plot(
    [y_val_original.min(), y_val_original.max()],
    [y_val_original.min(), y_val_original.max()],
    "r--", label="Perfect fit"
)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Predicted vs Actual (Validation Set)")
plt.legend()
plt.tight_layout()
plt.show()

# Residual plot
residuals = y_val_original - y_pred_val
plt.figure(figsize=(7, 4))
plt.scatter(y_pred_val, residuals, alpha=0.4, edgecolors="k", linewidths=0.3)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()

## 11. Feature Importance

LASSO shrinks irrelevant coefficients to exactly zero, effectively performing feature selection. The non-zero coefficients indicate the most predictive features.

In [ ]:
coefs = pd.Series(
    best_model.named_steps["lasso"].coef_,
    index=X.columns
)

selected = coefs[coefs != 0].sort_values(key=np.abs, ascending=False)

print(f"Features selected by LASSO: {len(selected)} out of {len(coefs)}")

# Plot top 15 features
top15 = selected.head(15)

colors = ["#d62728" if v < 0 else "#1f77b4" for v in top15.values]

plt.figure(figsize=(9, 6))
top15[::-1].plot(kind="barh", color=colors[::-1])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Top 15 LASSO Coefficients (standardized)")
plt.xlabel("Coefficient value")
plt.tight_layout()
plt.show()

print("\nTop 15 selected features:")
print(top15.to_string())

## 12. Kaggle Submission (Optional)

Generate predictions on the Kaggle test set and export as a submission CSV.

In [ ]:
# Predict on Kaggle test set
kaggle_preds_log = best_model.predict(X_kaggle)
kaggle_preds     = np.expm1(kaggle_preds_log)

# Build submission dataframe
submission = pd.DataFrame({
    "Id":        test["Id"],
    "SalePrice": kaggle_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved.")
print(submission.describe())